## Natural Language Processing Spring 2026
---
# End-to-End Pipeline: Pre-training, SFT, Checkpointing, and Evaluation

**Objective:** Build a robust data pipeline and two-phase training loop for our MiniMind Causal Language Model, save the resulting models, and evaluate them.

To ensure this notebook runs completely independently:
* **Step 1** downloads the required datasets from Hugging Face.
* **Step 2** explores the downloaded datasets.
* **Step 3** includes the full implementation of the `MiniMind` architecture.
* **Step 4** constructs both the `PretrainDataset` and the `SFTDataset`.
* **Step 5** implements a memory-efficient training loop featuring Gradient Accumulation, Clipping, and **Checkpoint Saving**.
* **Step 6** executes Phase 1 (Pre-training) and Phase 2 (SFT), saving checkpoints for both.
* **Step 7** introduces a generation function and performs a **Side-by-Side Computation** comparing the Pre-trained model against the SFT model.

---

### Two-Phase LLM Training at a Glance

Training a modern Language Model typically proceeds in two distinct phases:

| Phase | Objective | Data | Loss computed on |
|---|---|---|---|
| **1 — Pre-training** | Learn language structure and world knowledge | Unlabelled text (next-token prediction) | Every token |
| **2 — SFT (Supervised Fine-Tuning)** | Learn to follow instructions | Labelled conversation pairs | Assistant turns only |

This notebook demonstrates both phases end-to-end on small "mini" subsets so that the entire pipeline fits in a free Colab session.


## Step 0: Setup and Imports

### Library Overview

| Import | Role |
|---|---|
| `os`, `json` | File system navigation and JSON serialisation |
| `torch` | Core tensor library for all numeric computation |
| `torch.nn` | Building blocks: `Module`, `Linear`, `Embedding`, `LayerNorm`, … |
| `torch.nn.functional` | Stateless ops: `softmax`, `cross_entropy`, `silu`, … |
| `torch.optim` | Optimisers: AdamW, SGD, … |
| `torch.utils.data.Dataset` | Abstract base that forces you to implement `__len__` and `__getitem__` |
| `torch.utils.data.DataLoader` | Wraps a `Dataset` and produces batches, handles shuffling and parallel workers |
| `contextlib.nullcontext` | A no-op context manager; lets us write `with autocast_ctx:` even when autocast is disabled on CPU |
| `huggingface_hub.hf_hub_download` | Downloads individual files from the Hugging Face Hub |
| `datasets.load_dataset` | Loads JSONL / CSV / Parquet files as an Arrow-backed `Dataset` object with lazy streaming support |


In [36]:
!pip install -q huggingface_hub datasets

import os
import json
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torch.utils.data import Dataset, DataLoader
import math
import time
from contextlib import nullcontext
from datasets import load_dataset, Features, Value
from huggingface_hub import hf_hub_download

## Step 1: Download Datasets from Hugging Face

### What is JSONL?

**JSONL** (JSON Lines) stores one JSON object per line. This makes it trivial to stream very large files without loading everything into RAM:

```
{"text": "The capital of France is Paris."}
{"text": "Water is composed of hydrogen and oxygen."}
```

### Pre-training data vs. SFT data

| File | Format | Purpose |
|---|---|---|
| `pretrain_t2t_mini.jsonl` | `{"text": "..."}` | Plain text — model learns to continue any sentence |
| `sft_t2t_mini.jsonl` | `{"conversations": [{role, content}, ...]}` | Conversation pairs — model learns to respond as an assistant |

`hf_hub_download` fetches a single file from a Hugging Face dataset repository and places it in `local_dir`. Under the hood it uses the same caching layer as `snapshot_download`, so repeated runs skip the network request.


In [37]:
# Define the Hugging Face repository and the files we want to download
repo_id = "jingyaogong/minimind_dataset"
files_to_download = [
    "pretrain_t2t_mini.jsonl",
    "sft_t2t_mini.jsonl"
]

# Create the target directory as expected by the pipeline
dataset_dir = "./dataset"
os.makedirs(dataset_dir, exist_ok=True)

print("Downloading datasets from Hugging Face...")
for file_name in files_to_download:
    print(f"Fetching {file_name}...")
    hf_hub_download(
        repo_id=repo_id,
        filename=file_name,
        repo_type="dataset",
        local_dir=dataset_dir
    )

print("\nAll datasets successfully downloaded to the './dataset' directory!")

Fetching pretrain_t2t_mini.jsonl...
Fetching sft_t2t_mini.jsonl...

All datasets successfully downloaded to the './dataset' directory!


## Step 2: Dataset Exploration and Statistics

Before writing a single line of training code it is good practice to understand the data:

* **Sample count** — how many training examples exist?
* **Field names** — do the keys match what the `Dataset` class expects?
* **Token budget** — how long is a typical sample compared to `max_length`?

Inspecting a few examples by eye also catches data-quality issues (garbled Unicode, empty strings, wrong schema) early.


In [38]:
dataset_files = [
    "./dataset/pretrain_t2t_mini.jsonl",
    "./dataset/sft_t2t_mini.jsonl"
]

print("--- Dataset Statistics & Examples ---\n")

for file_path in dataset_files:
    if not os.path.exists(file_path):
        continue

    count = 0
    first_sample = None
    with open(file_path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i == 0: first_sample = json.loads(line)
            count += 1

    print(f"📌 Dataset: {os.path.basename(file_path)}")
    print(f"Total Samples: {count:,}")
    print("Example Sample:")
    print(json.dumps(first_sample, ensure_ascii=False, indent=2))
    print("-" * 50 + "\n")

--- Dataset Statistics & Examples ---

📌 Dataset: pretrain_t2t_mini.jsonl
Total Samples: 1,270,238
Example Sample:
{
  "text": "给我生成一首有关秋天的诗歌。秋日早晨，清风拂面。\n金色的叶子，似火在燃烧。\n露珠晶莹，如珍珠般美丽。\n秋的气息，弥漫在空气中。\n余音袅袅，如鸟儿的歌唱。\n美丽的秋天，是大自然的馈赠。帮我想一些创意，给即将到来的公司年会准备节目。一些节目比如能否请一位表演者为我们表演一曲钢琴曲，或者请一位小提琴手为大家演奏一首古典曲目。如果想要画面更具有视觉冲击力，可以安排一个魔术师或者杂技演员的表演。另外，也可以设计一些小游戏或者有奖竞猜来增添活动的趣味性，这些小游戏可以和公司的文化、发展历程等相关。请问给我讲一个清净的法则。无为而治是一项清净的法则。即在处理问题时，不要强行干预，反而尽可能地减少干预，坚持自然的发展趋势。比如，让植物自然生长，照顾它们就行了，不需要过度地修剪和整齐地排列。类似地，让人们自由发展，而不是通过繁琐的管制、政策干涉等方式，去要求、指导人们的行为。这种无为而治的法则旨在维持一个有秩序的、和平的治理状态，却不需要大规模的干预和指挥。"
}
--------------------------------------------------

📌 Dataset: sft_t2t_mini.jsonl
Total Samples: 905,718
Example Sample:
{
  "conversations": [
    {
      "role": "user",
      "content": "你背后的模型是哪个版本？它由谁开发？"
    },
    {
      "role": "assistant",
      "content": "我是由jingyaogong开发的高效小参数AI模型。"
    },
    {
      "role": "user",
      "content": "你模型的训练数据来源是什么？"
    },
    {
      "role": "assistant",
      "content": "我的训练数据涵盖多领域，确

**Reading the output above:**

* `pretrain_t2t_mini.jsonl` — each record has a `"text"` field containing a raw sentence or paragraph. This is the standard "text prediction" format.
* `sft_t2t_mini.jsonl` — each record has a `"conversations"` list. Every element in the list is a dict with `"role"` (`user` / `assistant`) and `"content"`. This mirrors the OpenAI chat format.

The `_mini` suffix means these are tiny subsets (a few thousand samples each), sized to let you iterate quickly on Colab's free tier.


## Step 3: MiniMind Architecture (Standalone)

In [39]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# ==========================================
# 1. Configuration & Normalization
# ==========================================
class MiniMindConfig:
    """
    Configuration class holding the hyperparameters for the MiniMind model.
    Defines dimensions, layer counts, heads, and other architectural details.
    """
    def __init__(self):
        self.vocab_size = 6400
        self.hidden_size = 256  # Reduced for faster demonstration
        self.num_hidden_layers = 2
        self.num_attention_heads = 8
        self.num_key_value_heads = 4
        self.head_dim = self.hidden_size // self.num_attention_heads
        self.intermediate_size = 1024
        self.max_position_embeddings = 2048
        self.rms_norm_eps = 1e-6
        self.dropout = 0.0
        self.tie_word_embeddings = True

class RMSNorm(nn.Module):
    """
    Root Mean Square Normalization (RMSNorm).
    A variant of LayerNorm that is computationally cheaper and often used in modern LLMs (like LLaMA).
    It normalizes the activations based on their root mean square.
    """
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        normed = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return (self.weight * normed.float()).type_as(x)

# ==========================================
# 2. Rotary Position Embeddings (RoPE)
# ==========================================
def precompute_freqs_cis(dim: int, end: int, rope_base: float = 1e6):
    """
    Precomputes the cosine and sine frequencies for Rotary Position Embeddings (RoPE).
    RoPE injects positional information directly into the Query and Key matrices.
    """
    freqs = 1.0 / (rope_base ** (torch.arange(0, dim, 2).float() / dim))
    t = torch.arange(end, device=freqs.device)
    freqs = torch.outer(t, freqs).float()
    freqs_cos = torch.cat([torch.cos(freqs), torch.cos(freqs)], dim=-1)
    freqs_sin = torch.cat([torch.sin(freqs), torch.sin(freqs)], dim=-1)
    return freqs_cos, freqs_sin

def apply_rotary_pos_emb(q, k, cos, sin):
    """
    Applies the precomputed rotary frequencies to the Query and Key tensors.
    """
    def rotate_half(x):
        half_dim = x.shape[-1] // 2
        return torch.cat((-x[..., half_dim:], x[..., :half_dim]), dim=-1)
    cos_unsqueezed = cos.unsqueeze(0).unsqueeze(2)
    sin_unsqueezed = sin.unsqueeze(0).unsqueeze(2)
    q_embed = (q * cos_unsqueezed) + (rotate_half(q) * sin_unsqueezed)
    k_embed = (k * cos_unsqueezed) + (rotate_half(k) * sin_unsqueezed)
    return q_embed, k_embed

# ==========================================
# 3. Attention & FeedForward Modules
# ==========================================
def repeat_kv(x: torch.Tensor, n_rep: int) -> torch.Tensor:
    """
    Repeats Key and Value heads to match the number of Query heads.
    Used in Grouped-Query Attention (GQA).
    """
    bs, slen, num_kv_heads, head_dim = x.shape
    if n_rep == 1: return x
    x_expanded = x[:, :, :, None, :].expand(bs, slen, num_kv_heads, n_rep, head_dim)
    return x_expanded.reshape(bs, slen, num_kv_heads * n_rep, head_dim)

class Attention(nn.Module):
    """
    Grouped-Query Attention mechanism.
    Projects inputs into Queries, Keys, and Values, applies RoPE, computes attention scores,
    and then projects the context vector back to the hidden size.
    """
    def __init__(self, config):
        super().__init__()
        self.n_local_heads = config.num_attention_heads
        self.n_local_kv_heads = config.num_key_value_heads
        self.n_rep = self.n_local_heads // self.n_local_kv_heads
        self.head_dim = config.head_dim
        self.q_proj = nn.Linear(config.hidden_size, self.n_local_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(config.hidden_size, self.n_local_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(config.hidden_size, self.n_local_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(self.n_local_heads * self.head_dim, config.hidden_size, bias=False)
        self.q_norm = RMSNorm(self.head_dim, eps=config.rms_norm_eps)
        self.k_norm = RMSNorm(self.head_dim, eps=config.rms_norm_eps)

    def forward(self, x, position_embeddings):
        bsz, seq_len, _ = x.shape
        xq, xk, xv = self.q_proj(x), self.k_proj(x), self.v_proj(x)
        xq = xq.view(bsz, seq_len, self.n_local_heads, self.head_dim)
        xk = xk.view(bsz, seq_len, self.n_local_kv_heads, self.head_dim)
        xv = xv.view(bsz, seq_len, self.n_local_kv_heads, self.head_dim)
        xq, xk = self.q_norm(xq), self.k_norm(xk)
        cos, sin = position_embeddings
        xq, xk = apply_rotary_pos_emb(xq, xk, cos, sin)
        xq = xq.transpose(1, 2)
        xk = repeat_kv(xk, self.n_rep).transpose(1, 2)
        xv = repeat_kv(xv, self.n_rep).transpose(1, 2)
        scores = (xq @ xk.transpose(-2, -1)) / math.sqrt(self.head_dim)
        mask = torch.full((seq_len, seq_len), float("-inf"), device=scores.device).triu(1)
        scores = scores + mask
        attention_output = F.softmax(scores, dim=-1) @ xv
        attention_output = attention_output.transpose(1, 2).contiguous().reshape(bsz, seq_len, -1)
        return self.o_proj(attention_output)

class FeedForward(nn.Module):
    """
    SwiGLU FeedForward Network.
    Projects the hidden states to a larger intermediate size, applies the SiLU activation
    gated by another projection, and down-projects back to the hidden size.
    """
    def __init__(self, config):
        super().__init__()
        self.gate_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.up_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.down_proj = nn.Linear(config.intermediate_size, config.hidden_size, bias=False)
    def forward(self, x):
        gated_features = F.silu(self.gate_proj(x)) * self.up_proj(x)
        return self.down_proj(gated_features)

# ==========================================
# 4. Transformer Block & LM Wrapper
# ==========================================
class MiniMindBlock(nn.Module):
    """
    A single Transformer layer containing RMSNorms, Attention, and the FeedForward network.
    """
    def __init__(self, config):
        super().__init__()
        self.self_attn = Attention(config)
        self.input_layernorm = RMSNorm(config.hidden_size)
        self.post_attention_layernorm = RMSNorm(config.hidden_size)
        self.mlp = FeedForward(config)
    def forward(self, x, position_embeddings):
        x = x + self.self_attn(self.input_layernorm(x), position_embeddings)
        x = x + self.mlp(self.post_attention_layernorm(x))
        return x

class MiniMindForCausalLM(nn.Module):
    """
    The full Causal Language Model wrapping the embedding layer, Transformer blocks, and the language modeling head.
    Computes cross-entropy loss if labels are provided.
    """
    def __init__(self, config):
        super().__init__()
        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size)
        self.layers = nn.ModuleList([MiniMindBlock(config) for _ in range(config.num_hidden_layers)])
        self.norm = RMSNorm(config.hidden_size)
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)
        if config.tie_word_embeddings:
            self.embed_tokens.weight = self.lm_head.weight

    def forward(self, input_ids, position_embeddings, labels=None):
        x = self.embed_tokens(input_ids)
        for layer in self.layers:
            x = layer(x, position_embeddings)
        x = self.norm(x)
        logits = self.lm_head(x)
        loss = None
        if labels is not None:
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=-100
            )
        return logits, loss

## Step 4: The Data Pipelines (Pre-training and SFT)

### The PyTorch `Dataset` Contract

Any class that inherits from `torch.utils.data.Dataset` must implement exactly two methods:

```python
def __len__(self) -> int:          # total number of samples
def __getitem__(self, index) -> ...: # return one sample by index
```

`DataLoader` calls these methods to produce shuffled mini-batches. The `Dataset` stores the raw data; the `DataLoader` handles batching, shuffling, and multiprocess loading.

---

### 4-A  Pre-training Dataset — Next-Token Prediction

The goal of pre-training is **next-token prediction**: given tokens $t_1, t_2, \ldots, t_n$ the model must predict $t_2, t_3, \ldots, t_{n+1}$. This is a **self-supervised** objective — the "labels" come directly from the input sequence shifted by one position.

#### Tokenisation pipeline

```
raw text  →  tokenizer  →  token IDs  →  pad to max_length
```

Key choices made below:

| Step | Detail |
|---|---|
| `add_special_tokens=False` | We add BOS and EOS ourselves so we control their position exactly |
| Prepend `bos_token_id` | Marks the beginning of a sequence; the model learns "start here" |
| Append `eos_token_id` | Marks the end; without it the model never learns to stop generating |
| Pad to `max_length` | All sequences in a batch must be the same length for efficient GPU tensor ops |
| `labels = input_ids.clone()` | For pre-training, every position is a supervision signal |
| `labels[pad] = -100` | `F.cross_entropy` ignores positions labelled `-100` (its default `ignore_index`), so padding does not contribute to the loss |

#### Why `labels = input_ids`?

During the forward pass the model predicts `logits` of shape `(batch, seq_len, vocab_size)`. The loss computation **shifts** by one:

```
input_ids:  [BOS, t1, t2, t3, EOS, PAD, ...]
labels:     [BOS, t1, t2, t3, EOS, PAD, ...]
                 ↑ at position i we predict labels[i+1]
```

The shift (`logits[..., :-1, :]` vs `labels[..., 1:]`) is done inside `MiniMindForCausalLM.forward`.

---

### 4-B  SFT Dataset — Instruction Following

SFT teaches the model to play the role of an assistant. The critical difference from pre-training is **selective loss masking**: we set all user-turn tokens to `-100` so the loss is **only** computed on the assistant's reply.

#### Why mask the user prompt?

If we trained on both user and assistant tokens, the model would also try to learn to generate user-style questions — the opposite of what we want. By masking the prompt we force the model to condition on the user's words and learn to produce high-quality responses.

```
conversation: [user: "What is 2+2?", assistant: "4."]
input_ids:    [BOS, "What", "is", "2", "+", "2", "?", "4", ".", EOS]
labels:       [ -100, -100,  -100, -100, -100, -100, -100, "4", ".", EOS]
                ←————————— masked ————————————→  ←— learned —→
```

The `generate_labels` helper below implements this masking logic (simplified for this demo).

#### `load_dataset` and the `Features` schema

`load_dataset('json', data_files=..., features=...)` reads the JSONL file into an Apache Arrow table.  
Passing an explicit `Features` object tells the library exactly what types to expect, which prevents slow type-inference and avoids schema errors when nested lists or nullable fields are present.


In [40]:
import torch
from torch.utils.data import Dataset
from datasets import load_dataset, Features, Value

# ==========================================
# 1. Pretrain Dataset
# ==========================================
class PretrainDataset(Dataset):
    """
    Dataset for Phase 1: Pre-training.
    Objective: Predict the next token (Standard Language Modeling).
    We train on all tokens, including user text.
    """
    def __init__(self, data_path, tokenizer, max_length=512):
        super().__init__()
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.samples = load_dataset('json', data_files=data_path, split='train')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        sample = self.samples[index]
        tokens = self.tokenizer(str(sample['text']), add_special_tokens=False, max_length=self.max_length - 2, truncation=True).input_ids
        tokens = [self.tokenizer.bos_token_id] + tokens + [self.tokenizer.eos_token_id]
        input_ids = tokens + [self.tokenizer.pad_token_id] * (self.max_length - len(tokens))

        input_ids = torch.tensor(input_ids, dtype=torch.long)
        labels = input_ids.clone()
        labels[input_ids == self.tokenizer.pad_token_id] = -100 # Mask padding to ignore in loss computation
        return input_ids, labels

# ==========================================
# 2. SFT Dataset (Supervised Fine-Tuning)
# ==========================================
class SFTDataset(Dataset):
    """
    Dataset for Phase 2: Supervised Fine-Tuning.
    Objective: Learn to follow instructions in a conversation format.
    We mask out the user prompts so the loss is ONLY computed on the assistant's replies.
    """
    def __init__(self, jsonl_path, tokenizer, max_length=1024):
        super().__init__()
        self.tokenizer = tokenizer
        self.max_length = max_length
        features = Features({'conversations': [{'role': Value('string'), 'content': Value('string'), 'reasoning_content': Value('string'), 'tools': Value('string'), 'tool_calls': Value('string')}]})
        self.samples = load_dataset('json', data_files=jsonl_path, split='train', features=features)
        self.bos_id = [tokenizer.bos_token_id]
        self.eos_id = [tokenizer.eos_token_id]

    def __len__(self):
        return len(self.samples)

    def generate_labels(self, input_ids):
        """
        Masks out user prompts (-100) so only the assistant's response is learned.
        For this simple demo, we simulate masking by only predicting the second half.
        """
        labels = [-100] * len(input_ids)
        start_assistant = len(input_ids) // 2
        for j in range(start_assistant, len(input_ids)):
            if input_ids[j] != self.tokenizer.pad_token_id:
                labels[j] = input_ids[j]
        return labels

    def __getitem__(self, index):
        sample = self.samples[index]
        prompt = str(sample['conversations'])
        input_ids = self.tokenizer(prompt, add_special_tokens=False, max_length=self.max_length, truncation=True).input_ids
        input_ids += [self.tokenizer.pad_token_id] * (self.max_length - len(input_ids))
        labels = self.generate_labels(input_ids)

        return torch.tensor(input_ids, dtype=torch.long), torch.tensor(labels, dtype=torch.long)

**Key PyTorch functions used above:**

| Function / class | What it does |
|---|---|
| `torch.tensor(data, dtype=torch.long)` | Creates a new 1-D tensor from a Python list of integers. `dtype=torch.long` (int64) is required by `nn.Embedding` |
| `labels.clone()` | Makes a deep copy of the tensor so that masking `labels` does not change `input_ids` |
| `tensor[mask] = -100` | Boolean indexing — sets all positions where `mask` is `True` to the sentinel value `-100` |
| `Dataset.select(range(N))` | HuggingFace method to take the first N rows (used in Step 6 to keep the demo fast) |


## Step 5: The Core Training Loop (with Checkpointing)

We have added a new `save_path` parameter to `train_epoch_demo`. At the end of each training phase, the model will dump its `state_dict` to disk, allowing us to load these distinct checkpoints later for evaluation.

---

### Training Techniques Explained

#### 1. Mixed-Precision Training (`torch.amp.autocast`)

Modern GPUs (Volta and later) have specialised hardware for **float16** arithmetic that is roughly 2–8× faster than float32 and uses half the memory. PyTorch's Automatic Mixed Precision (AMP) framework automatically casts eligible operations (matrix multiplications, convolutions) to float16 while keeping numerically sensitive operations (loss, normalisation) in float32.

```python
with torch.amp.autocast('cuda', dtype=torch.float16):
    logits, loss = model(...)  # key ops run in fp16
```

On CPU we use `nullcontext()` as a no-op so the same code path works without modification.

#### 2. GradScaler — preventing fp16 underflow

When gradients are computed in float16 they can **underflow** to zero (values smaller than ~6 × 10⁻⁵ are rounded to 0). `GradScaler` multiplies the loss by a large scale factor before the backward pass, then divides the gradients back before the optimiser step. It also automatically detects overflow (NaN/Inf) and skips the step if it occurs.

```python
scaler.scale(loss).backward()   # backward with scaled loss
scaler.unscale_(optimizer)      # unscale before clipping
scaler.step(optimizer)          # optimizer step (skipped on overflow)
scaler.update()                 # adjust scale factor for next iter
```

#### 3. Gradient Accumulation

Real-world LLM training uses very large batch sizes (often 512–4096 sequences) to get stable gradient estimates. When GPU memory is limited we simulate a large batch by accumulating gradients over `K` smaller mini-batches before taking an optimiser step:

```
effective_batch_size = batch_size × accumulation_steps
```

We divide the loss by `accumulation_steps` before the backward pass so that the accumulated gradient has the same magnitude as if we had computed it over the full effective batch in one shot.

#### 4. Gradient Clipping (`clip_grad_norm_`)

Early in training (or after a bad batch) gradients can become very large and cause the model parameters to jump to a bad region — the **exploding gradient** problem. `torch.nn.utils.clip_grad_norm_` rescales all gradients so that their global L2 norm does not exceed `max_norm`:

$$g \leftarrow g \cdot \min\left(1, \frac{\text{max\_norm}}{\|g\|_2}\right)$$

This must be called **after** `scaler.unscale_()` so gradients are in their true float32 scale.

#### 5. `optimizer.zero_grad(set_to_none=True)`

Setting `set_to_none=True` frees the gradient tensors entirely (instead of filling them with zeros), reducing peak memory usage by up to 50% during the accumulation steps where gradients are not needed.

#### 6. Checkpoint Saving (`torch.save`)

`model.state_dict()` returns a Python `OrderedDict` mapping parameter names to their current tensors. `torch.save` serialises this to disk using Python's `pickle` format. This is the standard way to save and restore model weights in PyTorch.


In [41]:
def train_epoch_demo(model, loader, optimizer, scaler, args, pos_embs, phase_name, save_path):
    device_type = "cuda" if "cuda" in args.device else "cpu"
    dtype = torch.float16
    autocast_ctx = nullcontext() if device_type == "cpu" else torch.amp.autocast('cuda', dtype=dtype)

    model.train()
    print(f"\n>>> Starting Phase: {phase_name} <<<")
    for step, (input_ids, labels) in enumerate(loader, start=1):
        input_ids = input_ids.to(args.device)
        labels = labels.to(args.device)

        with autocast_ctx:
            logits, loss = model(input_ids, position_embeddings=pos_embs, labels=labels)
            loss = loss / args.accumulation_steps

        scaler.scale(loss).backward()

        if step % args.accumulation_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        if step % args.log_interval == 0:
            print(f"[{phase_name}] Step {step}: Loss = {loss.item() * args.accumulation_steps:.4f}")

    if step % args.accumulation_steps != 0:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)

    # Save Model Checkpoint
    print(f"Saving {phase_name} checkpoint to {save_path}...")
    torch.save(model.state_dict(), save_path)
    print("Checkpoint saved successfully!")

## Step 6: End-to-End Execution (Phase 1 & Phase 2)

We will now simulate the entire LLM training lifecycle using the `_mini` subsets. We'll save `pretrain_model.pth` and `sft_model.pth` to disk.

---

### Design Decisions in This Cell

#### `TrainArgs` — hyperparameter container

Using a plain class (instead of a `dict` or `argparse.Namespace`) gives attribute-style access (`args.device`) and makes it easy to document each field. In production you would use `argparse` or a config library like `hydra`.

#### `AdamW` — why not plain `Adam`?

**AdamW** decouples the weight-decay regularisation from the gradient update. In standard Adam, L2 weight decay is erroneously mixed into the adaptive learning-rate scaling, which reduces its regularisation effect. AdamW fixes this, which is why it has become the default for transformer training.

#### Lower learning rate for SFT

Fine-tuning starts from a **pre-trained checkpoint** that already contains useful representations. A large learning rate would destroy this knowledge (catastrophic forgetting). The standard practice is to use a learning rate 5–10× smaller than the pre-training rate:

```python
for param_group in optimizer.param_groups:
    param_group['lr'] = 5e-5   # down from 5e-4
```

#### `MockTokenizer`

Because the full `sentencepiece` tokenizer is not loaded here, we use a deterministic mock that maps each character to a pseudo-token ID. This lets the pipeline run end-to-end without requiring the pre-trained vocabulary, at the cost of producing meaningless output text.

#### `GradScaler` with `enabled=...`

Passing `enabled=False` on CPU creates a scaler whose methods are all no-ops, so the same training code runs on both CPU and GPU without `if/else` branches.


In [42]:
class TrainArgs:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    epochs = 100
    learning_rate = 5e-4
    accumulation_steps = 2
    grad_clip = 1.0
    log_interval = 5
    max_seq_len = 32

args = TrainArgs()
config = MiniMindConfig()
os.makedirs("./checkpoints", exist_ok=True)

model = MiniMindForCausalLM(config).to(args.device)
pos_embs = precompute_freqs_cis(config.head_dim, args.max_seq_len)
pos_embs = (pos_embs[0].to(args.device), pos_embs[1].to(args.device))

class MockTokenizer:
    bos_token_id = 1
    eos_token_id = 2
    pad_token_id = 0
    def __call__(self, text, add_special_tokens=False, max_length=512, truncation=True):
        class Output:
            input_ids = [(ord(c) % 6300) + 10 for c in text][:max_length]
        return Output()
    def decode(self, token_ids):
        return "".join([chr(t - 10) if 32 <= (t-10) <= 126 else "*" for t in token_ids])

tokenizer = MockTokenizer()
optimizer = optim.AdamW(model.parameters(), lr=args.learning_rate)
scaler = torch.amp.GradScaler("cuda", enabled=("cuda" in args.device))

print("=== MiniMind End-to-End Training Simulation ===")

# --- PHASE 1: PRE-TRAINING ---
pretrain_path = "./dataset/pretrain_t2t_mini.jsonl"
if os.path.exists(pretrain_path):
    ds_pretrain = PretrainDataset(pretrain_path, tokenizer, max_length=args.max_seq_len)
    ds_pretrain.samples = ds_pretrain.samples.select(range(20)) # Increase this for better performance!
    loader_pretrain = DataLoader(ds_pretrain, batch_size=4, shuffle=True)

    for epoch in range(args.epochs):
        train_epoch_demo(model, loader_pretrain, optimizer, scaler, args, pos_embs,
                         phase_name=f"Pre-training (Epoch {epoch+1}/{args.epochs})",
                         save_path="./checkpoints/pretrain_model.pth")

# --- PHASE 2: SUPERVISED FINE-TUNING ---
sft_path = "./dataset/sft_t2t_mini.jsonl"
if os.path.exists(sft_path):
    ds_sft = SFTDataset(sft_path, tokenizer, max_length=args.max_seq_len)
    ds_sft.samples = ds_sft.samples.select(range(20)) # Increase this for better performance!
    loader_sft = DataLoader(ds_sft, batch_size=4, shuffle=True)

    # Lower learning rate for fine-tuning phase
    for param_group in optimizer.param_groups:
        param_group['lr'] = 5e-5

    for epoch in range(args.epochs):
        train_epoch_demo(model, loader_sft, optimizer, scaler, args, pos_embs,
                         phase_name=f"Supervised Fine-Tuning (Epoch {epoch+1}/{args.epochs})",
                         save_path="./checkpoints/sft_model.pth")

=== MiniMind End-to-End Training Simulation ===

>>> Starting Phase: Pre-training (Epoch 1/100) <<<
[Pre-training (Epoch 1/100)] Step 5: Loss = 8.6518
Saving Pre-training (Epoch 1/100) checkpoint to ./checkpoints/pretrain_model.pth...
Checkpoint saved successfully!

>>> Starting Phase: Pre-training (Epoch 2/100) <<<
[Pre-training (Epoch 2/100)] Step 5: Loss = 6.9804
Saving Pre-training (Epoch 2/100) checkpoint to ./checkpoints/pretrain_model.pth...
Checkpoint saved successfully!

>>> Starting Phase: Pre-training (Epoch 3/100) <<<
[Pre-training (Epoch 3/100)] Step 5: Loss = 6.0237
Saving Pre-training (Epoch 3/100) checkpoint to ./checkpoints/pretrain_model.pth...
Checkpoint saved successfully!

>>> Starting Phase: Pre-training (Epoch 4/100) <<<
[Pre-training (Epoch 4/100)] Step 5: Loss = 4.9101
Saving Pre-training (Epoch 4/100) checkpoint to ./checkpoints/pretrain_model.pth...
Checkpoint saved successfully!

>>> Starting Phase: Pre-training (Epoch 5/100) <<<
[Pre-training (Epoch 5/100)]

## Step 7: Side-by-Side Computation & Evaluation

With our checkpoints saved to disk, we can now initialize two separate models in memory—one loaded with the Pre-trained weights, and the other with the SFT weights.

We'll build a basic autoregressive text generation function and pass the same prompt to both models. Since they have been trained on different data objectives (word chain completion vs. instruction following), we can compute and compare their responses side-by-side.

---

### Autoregressive Text Generation

A causal language model is trained to estimate the probability distribution over the next token $P(t_{n+1} | t_1, \ldots, t_n)$. At inference time we **sample** from this distribution repeatedly to produce a full response:

```
prompt → [t1, t2, …, tn]
   step 1: run model → logits → pick next token → tn+1
   step 2: run model on [t1, …, tn, tn+1] → pick tn+2
   ...
```

Because each new token is appended to the input and fed back into the model, this is called **autoregressive** (self-referential) generation.

#### Greedy Decoding

`torch.argmax(logits[:, -1, :], dim=-1)` always picks the single most probable next token. This is the simplest decoding strategy:

* **Pros:** deterministic, no randomness, fast.
* **Cons:** can produce repetitive or suboptimal sequences (a locally greedy choice may not be globally optimal).

Alternative strategies include **beam search** (keep top-k partial sequences), **top-p / nucleus sampling** (sample from the smallest set of tokens whose cumulative probability ≥ p), and **temperature scaling** (divide logits by T before softmax to sharpen or flatten the distribution).

#### `torch.no_grad()` context

During inference we do not need to store the intermediate activations required for backpropagation. Wrapping generation inside `with torch.no_grad():` disables the autograd engine, which:

1. Reduces memory usage (no gradient buffers allocated).
2. Speeds up the forward pass slightly.

#### `model.load_state_dict(torch.load(path))`

`torch.load` deserialises the saved `OrderedDict` back into CPU memory. `load_state_dict` then copies each tensor into the corresponding parameter of the model. Using `weights_only=True` (PyTorch ≥ 2.0) is recommended in production to avoid arbitrary code execution from untrusted checkpoint files.

#### Why two separate model instances?

We instantiate `model_pretrain` and `model_sft` independently and load different checkpoints into each. This lets us run them truly in parallel (or sequentially in memory) and compare outputs under identical conditions — same prompt, same decoding strategy, different weights.


In [43]:
def generate_text(model, tokenizer, prompt, max_new_tokens=15):
    model.eval()
    input_ids = torch.tensor([tokenizer(prompt, max_length=args.max_seq_len).input_ids], dtype=torch.long).to(args.device)

    # Precompute RoPE for the max possible length during generation
    pos_embs = precompute_freqs_cis(config.head_dim, input_ids.size(1) + max_new_tokens)
    pos_embs = (pos_embs[0].to(args.device), pos_embs[1].to(args.device))

    with torch.no_grad():
        for _ in range(max_new_tokens):
            current_seq_len = input_ids.size(1)
            current_pos_embs = (pos_embs[0][:current_seq_len], pos_embs[1][:current_seq_len])

            logits, _ = model(input_ids, position_embeddings=current_pos_embs)
            # Greedy decoding: pick the token with the highest probability
            next_token = torch.argmax(logits[:, -1, :], dim=-1).unsqueeze(1)
            input_ids = torch.cat([input_ids, next_token], dim=1)

    return tokenizer.decode(input_ids[0].cpu().tolist())

print("\n=== Loading Models for Side-by-Side Evaluation ===")
# 1. Initialize Base Architecture for Both
model_pretrain = MiniMindForCausalLM(config).to(args.device)
model_sft = MiniMindForCausalLM(config).to(args.device)

# 2. Load the Checkpoints
if os.path.exists("./checkpoints/pretrain_model.pth") and os.path.exists("./checkpoints/sft_model.pth"):
    model_pretrain.load_state_dict(torch.load("./checkpoints/pretrain_model.pth"))
    model_sft.load_state_dict(torch.load("./checkpoints/sft_model.pth"))
    print("Checkpoints successfully loaded into memory.")

    # 3. Test with a sample prompt
    test_prompt = "Hello Assistant, can you help me?"
    print(f"\n[Prompt]: {test_prompt}\n")

    # Note: Because this notebook uses a severely truncated 20-sample dataset,
    # a micro 256-dim model, and a mock ASCII tokenizer, the output will look like
    # random characters. In a full run, this is where you observe the stark
    # behavioral difference between completion (Pretrain) and chatting (SFT).

    out_pretrain = generate_text(model_pretrain, tokenizer, test_prompt, max_new_tokens=10)
    out_sft = generate_text(model_sft, tokenizer, test_prompt, max_new_tokens=10)

    print("=== SIDE-BY-SIDE COMPUTATION ===")
    print("🤖 Pre-trained Model Output:")
    print(f"> {out_pretrain}\n")
    print("🤖 SFT Model Output:")
    print(f"> {out_sft}\n")
    print("==================================")
else:
    print("Checkpoints not found. Ensure Step 6 completed successfully.")


=== Loading Models for Side-by-Side Evaluation ===
Checkpoints successfully loaded into memory.

[Prompt]: Hello Assistant, can you help me?

=== SIDE-BY-SIDE COMPUTATION ===
🤖 Pre-trained Model Output:
> Hello Assistant, can you help me**********

🤖 SFT Model Output:
> Hello Assistant, can you help ment'**t'**t

